In [1]:
### Document structure
from langchain_core.documents import Document

In [3]:
doc=Document(
    page_content="this is the main page conetent i am using to create the rag",
    metadata={
        "source":"youtube",
        "pages":1,
        "author":"krish naik"
    }
)
doc

Document(metadata={'source': 'youtube', 'pages': 1, 'author': 'krish naik'}, page_content='this is the main page conetent i am using to create the rag')

In [4]:
import os
os.makedirs("../data/text_files",exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [2]:
from langchain_community.document_loaders import TextLoader
 
loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
doc=loader.load()
print(doc)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
class EmbeddingManger:
    """handles documnets embdedding generation using sentencetransformer"""
    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager 

        Args:
        model_name = hugging face model name
        """
        self.model_name = model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model :{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully . embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self,texts:List[str]) -> np.ndarray:
        """ 
        generate embedding for the list of text

        Args:
        Texts: List of text strings to embed

        Returns:
        numpy array of embedding with shape (len(text)), embedding_din

        """
        if not self.model:
            raise ValueError("model not loaded")
        
        print(f"generating embedding for {len(texts)} texts..")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"shape : {embeddings.shape}")
        return embeddings

### initialzie the embedding manager

embedding_manager= EmbeddingManger()
embedding_manager

Loading embedding model :all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 173.60it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully . embedding dimension: 384
